# brier — M4 benchmark: raw vs L0 vs L1 on banking20 (Qwen3-1.7B)

Same protocol and pins as the M3.4 run (`docs/results.md`), plus **L1 temperature scaling**:
T is fitted on L0 predictions for the 300-item labelled calibration split and applied to the
2,603 test items. L1 never changes the answer; it should fix the overconfidence L0 showed
(ECE ≈ 0.30).

**M4 acceptance:** L1 ECE lower than L0 (paired CI entirely below 0).

**Runtime:** *Runtime → Change runtime type → A100 GPU*. About 15–20 min (L1 adds ~10 %).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Configuration and install

In [ ]:
BRIER_COMMIT = "3988cb7e15b5c714a821f33e885472cb8618a6ce"  # adds L1 to the benchmark
MODEL = "Qwen/Qwen3-1.7B"
REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"  # same pinned model as M3.4
DTYPE = "bfloat16"  # A100/L4 support bf16; on a T4 use "float16"
BATCH = 64
OUT = "/content/results"

In [ ]:
!git clone -q https://github.com/mohdUwaish59/brier.git /content/brier
!git -C /content/brier checkout -q {BRIER_COMMIT}
!pip install -q -e "/content/brier[hf]"
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

## 3. Timing run (50 items)

First 50 test, pool and calibration items (L1 needs at least 50 labelled items).

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype {DTYPE} \
    --batch-size {BATCH} --levels raw,L0,L1 --limit 50 --n-resamples 200 --out /content/results_timing

In [ ]:
import glob
import json

with open(glob.glob("/content/results_timing/*.json")[0]) as f:
    timing = json.load(f)
w = timing["timing"]["wall_seconds"]
# raw scales with test items; L0 with pool + test; L1 with calibration items.
est = w["raw"] * 2603 / 50 + w["L0"] * (500 + 2603) / 100 + w["L1"] * 300 / 50
print(f"timing run: {w}")
print(f"estimated full run: ~{est / 60:.0f} min")

## 4. Full run

Keep this tab open. Results are written after each level (raw, L0, L1).

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype {DTYPE} \
    --batch-size {BATCH} --levels raw,L0,L1 --out {OUT}

## 5. Results

In [ ]:
import glob
import json

with open(glob.glob(f"{OUT}/*.json")[0]) as f:
    res = json.load(f)


def fmt(v):
    return f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]"


cal = res["calibration"]["L1"]
rev = res["model"]["revision"] or "unpinned"
print(f"model {res['model']['id']}@{rev[:8]}  brier {res['git_commit']}")
print(
    f"n_test={res['n_test']} n_pool={res['n_pool']} n_calib={cal['n_calib']} "
    f"T={cal['temperature']:.3f} (reversed {cal['temperature_reversed']:.3f})"
)
print()
print("| metric | raw | L0 | L1 | L1 - L0 (paired) |")
print("|---|---|---|---|---|")
for k in res["levels"]["raw"]:
    cols = [fmt(res["levels"][lv][k]) for lv in ("raw", "L0", "L1")]
    diff = fmt(res["comparisons"]["L1_minus_L0"][k])
    print(f"| {k} | {' | '.join(cols)} | {diff} |")
print()
print("wall seconds:", res["timing"]["wall_seconds"])

# M4 acceptance: L1 ECE lower than L0, paired CI entirely below zero.
ok = res["comparisons"]["L1_minus_L0"]["ece"][2] < 0
print("M4 acceptance (L1 ECE < L0, paired CI below 0):", "PASS" if ok else "FAIL")

## 6. Download the results (JSON + per-item `.npz`, no state text)

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/brier_m4_results", "zip", OUT)
files.download("/content/brier_m4_results.zip")